In [7]:
# import libraries
import re
import heapq
import string
from tqdm import tqdm
import numpy as np
import pandas as pd
from urllib.request import urlopen

# if you need to plot anything
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt

# Stopword dictionary
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk import download, wordnet
nltk.download('omw-1.4')
download('punkt')
nltk.download('punkt_tab')
download('stopwords')
download("wordnet")



# For stemming
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
cv = CountVectorizer() # the Bag of Words
tfidf = TfidfVectorizer()

# let's import ML algorithms for classification
from sklearn.svm import SVC
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split as tts, KFold, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score as ac, confusion_matrix as cm, ConfusionMatrixDisplay as CMD

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dvasiliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/dvasiliu/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /home/dvasiliu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/dvasiliu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dvasiliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/dvasiliu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
## <font color='blue'> Application to the Wine Ratings Dataset </font>
# the first example uses only n observations
wine_data = pd.read_csv('https://github.com/dvasiliu/AML/blob/main/Data%20Sets/winemagdata130kv2.csv?raw=true',quoting=2)
wines = wine_data[["description","points"]]
# wines_subset = wines.sample(5000,random_state=1234).reset_index(drop=True)
corpus = []

In [4]:
wines

,description,points
0,"Aromas include tropical fruit, broom, brimston...",87.0
1,"This is ripe and fruity, a wine that is smooth...",87.0
2,"Tart and snappy, the flavors of lime flesh and...",87.0
3,"Pineapple rind, lemon pith and orange blossom ...",87.0
4,"Much like the regular bottling from 2012, this...",87.0
...,...,...
129966,Notes of honeysuckle and cantaloupe sweeten th...,90.0
129967,Citation is given as much as a decade of bottl...,90.0
129968,Well-drained gravel soil gives this wine its c...,90.0
129969,"A dry style of Pinot Gris, this is crisp with ...",90.0


In [5]:
wines.iloc[129966,0]

"Notes of honeysuckle and cantaloupe sweeten this deliciously feather-light spätlese. It's intensely juicy, quenching the palate with streams of tart tangerine and grapefruit acidity, yet wraps up with a kiss of honey and peach."

## Goals: Apply BoW and TFIDF to Perform Sentiment Analysis

In [3]:
wines_subset = wines["description"]

In [7]:
wines_subset

0         Aromas include tropical fruit, broom, brimston...
1         This is ripe and fruity, a wine that is smooth...
2         Tart and snappy, the flavors of lime flesh and...
3         Pineapple rind, lemon pith and orange blossom ...
4         Much like the regular bottling from 2012, this...
                                ...                        
129966    Notes of honeysuckle and cantaloupe sweeten th...
129967    Citation is given as much as a decade of bottl...
129968    Well-drained gravel soil gives this wine its c...
129969    A dry style of Pinot Gris, this is crisp with ...
129970    Big, rich and off-dry, this is powered by inte...
Name: description, Length: 129971, dtype: object

In [8]:
# you can do this but you will be frustated with the runtime!!!
prep_reviews = []
for i in tqdm(range(0,len(wines))):
    wine_descriptions = re.sub('[^a-zA-Z0-9 ]','',wines["description"][i])
    wine_descriptions=wine_descriptions.lower()
    wine_descriptions = wine_descriptions.split()
    wine_descriptions = [word for word in wine_descriptions if not word in set(stopwords.words('english'))]
    stemmer = PorterStemmer()
    wine_descriptions = [stemmer.stem(word) for word in wine_descriptions]
    wine_descriptions = " ".join(wine_descriptions)
    prep_reviews.append(wine_descriptions)

100%|██████████| 129971/129971 [02:36<00:00, 832.88it/s]


In [9]:
prep_reviews[129966]

'note honeysuckl cantaloup sweeten delici featherlight sptlese intens juici quench palat stream tart tangerin grapefruit acid yet wrap kiss honey peach'

In [3]:
def fast_preprocess(texts, lemmatize=True):

    # Fast text preprocessing function that handles:
    # - Lowercasing
    # - Punctuation removal
    # - Stopword removal
    # - Lemmatization (optional)

    # Arguments:
    #     texts: List of strings or single string to process
    #     lemmatize: Whether to perform lemmatization

    # Returns:
    #     Processed texts (list or string, matching input type)

    # Handle single string input
    single_input = isinstance(texts, str)
    if single_input:
        texts = [texts]



    stop_words = set(stopwords.words('english'))

    # Create translator for punctuation removal (faster than regex for this task)
    # this is one good trick that speeds the removal of punctuation
    translator = str.maketrans('', '', string.punctuation)

    # Initialize lemmatizer if needed
    lemmatizer = WordNetLemmatizer() if lemmatize else None

    processed_texts = []
    for text in texts:
        # Lowercase
        text = text.lower()

        # Remove punctuation
        text = text.translate(translator)

        # Tokenize (needed for both stopword removal and lemmatization)
        tokens = text.split()

        # Remove stopwords
        tokens = [token for token in tokens if token not in stop_words]

        # Lemmatize if requested
        if lemmatize:
            tokens = [lemmatizer.lemmatize(token) for token in tokens]

        # Rejoin into text
        processed_texts.append(' '.join(tokens))

    # Return in same format as input
    return processed_texts[0] if single_input else processed_texts

In [4]:
prep_reviews_lem = fast_preprocess(wines['description']) 

In [12]:
prep_reviews_lem[129966]

'note honeysuckle cantaloupe sweeten deliciously featherlight spätlese intensely juicy quenching palate stream tart tangerine grapefruit acidity yet wrap kiss honey peach'

## The Sentiment Analysis

In [5]:
x = prep_reviews_lem
y = wines['points'].values

In [13]:
model = ElasticNet(alpha=0.001,l1_ratio=0.75,max_iter=5000)
x_prep = tfidf.fit_transform(x)
model.fit(x_prep,y)
model.score(x_prep,y)

0.49096191838916114

In [16]:
len(y)

129971

In [17]:
# we want to ethically create a train/test situation 
# after this is understood we will do a complete K_Fold

In [22]:
y.max()

np.float64(100.0)

In [9]:
# if we treat the sentiment analysis as classification we need to bin the y values
# let's create two categories, such as excellent (90<=y<=100) and good (80<=y<=89)
yb = (y>=90) + 0

In [26]:
yb

array([0, 0, 0, ..., 1, 1, 1], shape=(129971,))

In [32]:
# ethical train/test handling means we apply tfidf or bow after the traintestsplit
xtrain, xtest, ytrain, ytest = tts(prep_reviews_lem,yb,test_size=0.3,shuffle=True,random_state=301)

In [29]:
xtrain_vec = tfidf.fit_transform(xtrain)
xtest_vec = tfidf.transform(xtest)

In [33]:
# define a classification model
model = LogisticRegression(C=100,solver='saga',max_iter=2000)
model.fit(xtrain_vec,ytrain)
model.score(xtest_vec,ytest)

0.8347866228970046

In [34]:
# for a complete K-Fold cross-validation I suggest using a pipeline and a cross-val-score for Stratified K_Fold

In [7]:
from sklearn.pipeline import  Pipeline

In [40]:
pipe = Pipeline([['TfIDF',tfidf],['Logistic Regression',model]])
skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=301)
cross_val_score(pipe,prep_reviews_lem,yb,cv=skf,scoring='accuracy')

array([0.83512214, 0.83892437, 0.83850119, 0.83965531, 0.83434639])

In [ ]:
# what if we try a Support Vector Machine Model
model = SVC(kernel='rbf',gamma=0.001)
pipe = Pipeline([['TfIDF',tfidf],['SVC',model]])
skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=301)
cross_val_score(pipe,prep_reviews_lem,yb,cv=skf,scoring='accuracy')